## Chatbot Notebook

In [24]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

In [25]:
PAPER_DIR = "papers"

In [27]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """Search for papers on arXiv based on a topic. and store their information
    
    Args:
        topic (str): The topic to search for.
        max_results (int): The maximum number of results to retrive (default: 5).
        
    Returns:
        List[str]: A list of paper IDs found in the search
    """
    
    # use arxiv library to search for papers
    client = arxiv.Client(page_size=max_results, delay_seconds=3.0, num_retries=3)
    
    # Search for the most relevant articles matching the querid topic and get the top results
    search = arxiv.Search(
        query = topic, 
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
        )
    
    papers = client.results(search)
    
    # Create the papers directory if it doesn't exist
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")
    
    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}
        
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id()) 
        paper_data = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date()),
        }
        papers_info[paper.get_short_id()] = paper_data
        
    # Save the updated papers info back to the JSON file
    
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
        
    print(f"Results are saved in: {file_path}")

    return paper_ids

In [28]:
search_papers("computers")


Results are saved in: papers/computers/papers_info.json


['1312.3300v1', '2207.05241v1', '2603.19778v1', '2601.11095v1', '2012.10468v1']

In [29]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper acros all topic directories.
    
    args:
        paper_id (str): The ID of the paper to search for.
    returns:
        JSPM string with paper information if found, otherwise a message indicating the paper was not found.
    """
    
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.exists(file_path):
                try:
                 with open(file_path, "r") as json_file:
                    papers_info = json.load(json_file)
                    if paper_id in papers_info:
                        return json.dumps(papers_info[paper_id], indent=2)
                except (json.JSONDecodeError, FileNotFoundError) as e:
                    print(f"Error reading {file_path}: {e}")
                    continue
    return f"There is no saved information related to the paper with ID: {paper_id}"


In [30]:
extract_info("2603.19778v1")

'{\n  "title": "Uniform Maximum Projection Designs for Computer Experiments",\n  "authors": [\n    "Miroslav Vo\\u0159echovsk\\u00fd",\n    "Jan Ma\\u0161ek"\n  ],\n  "summary": "Space-filling experimental designs are widely used in engineering computer experiments, where only a limited number of expensive model evaluations can be afforded. Distance-based designs such as Maximin or Minimax ensure global space-filling, while Latin hypercube sampling enforces uniform one-dimensional projections, yet neither guarantees uniformity in lowdimensional subspaces. Maximum Projection (MaxPro) designs were introduced to improve uniformity in low-dimensional subspaces, yet their original formulation relies on the Euclidean distance and may induce systematic density distortions in bounded domains. We demonstrate that the standard MaxPro criterion leads to statistically non-uniform sampling, resulting in undersampling of corner regions and biased Monte Carlo estimates.\\n  To remedy this issue, we i

## Tool Schema

In [31]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

In [32]:
mapping_tool_functions = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    result = mapping_tool_functions[tool_name](**tool_args)
    
    if result is None:
        return "The operation completed but no result returned from the tool."
    
    elif isinstance(result, list): 
        result = ", ".join(result)
        
    elif isinstance(result, dict):
        # convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
        
    else:
        # For any other type, convert using str()
        result = str(result)
    
    return result

## Chatbot code

In [33]:
load_dotenv()
client = anthropic.Anthropic()

In [34]:
def process_query(query):
    
    messages= [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens =2024,
                                      model = 'claude-sonnet-4-6',
                                      tools = tools,
                                      messages = messages
                                      )
    
    process_query = True
    while process_query:
        assistant_content = []
        
        for content in response.content:
            if content.type == 'text':
                print(content.text)
                assistant_content.append(content)
                if len(response.content) == 1:
                    process_query = False
            
            elif content.type == 'tool_use':
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with arguments: {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({'role': 'user',
                                 "content": [
                                 {
                                        "type": "tool_result",
                                        "tool_use_id": tool_id,
                                        "content": result
                                 }
                                 ]
                })
                response = client.messages.create(max_tokens =2024,
                                                    model = 'claude-sonnet-4-6',
                                                    tools = tools,
                                                    messages = messages)
                
                if len(response.content) == 1 and response.content[0].type == 'text':
                    print(response.content[0].text)
                    process_query = False

## Chat Loop

In [35]:
def chat_loop():
    print("Welcome to the arXiv paper search chatbot! Type your queries or 'quit' to exit.")
    
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                print("Goodbye!")
                break
            
            process_query(query)
            print ("\n--- End of response ---")
        except Exception as e:
            print(f"An error occurred: {str(e)}")

In [36]:
chat_loop()

Welcome to the arXiv paper search chatbot! Type your queries or 'quit' to exit.
Hello! I think you might mean **JARVIS** 😄 — like the AI assistant from Iron Man! I'm your AI assistant, here to help you with a variety of tasks.

Here's what I can help you with:

* 🔍 **Search for research papers** on arXiv on any topic you're interested in
* 📄 **Extract detailed information** about specific papers using their ID
* 💡 **Answer questions** and provide explanations on various subjects
* 🧠 **Brainstorm ideas** or help you think through problems

---

**What would you like to do today?** Just say the word and I'm on it! 🚀

--- End of response ---
I'm not able to help with that topic. This assistant is designed to help with **academic and research-related questions**, specifically for searching and exploring scholarly papers on arXiv.

Here are some examples of topics I *can* help you with:

* 🔬 **Science** – physics, biology, chemistry
* 💻 **Computer Science** – AI, machine learning, algorithm